# Lezione 13 — Consumare API con `requests`

## Obiettivi della lezione

- Capire il ciclo **request / response** di HTTP: cosa mandi, cosa ti rispondono.
- Fare richieste GET e POST con la libreria `requests` da Python.
- Leggere lo `status_code`, gli `headers` e il corpo di una risposta.
- Convertire una risposta JSON in un DataFrame di pandas, incluso il caso di JSON **annidato** con `pd.json_normalize()`.
- Consumare un'API pubblica reale (senza autenticazione) durante la lezione.

> Nota tecnica: questo notebook fa richieste HTTP reali verso internet. Su Google Colab `requests` è già installato. Se lo esegui in Jupyter locale e non ce l'hai, decommenta `# !pip install requests` nella prima cella di codice.

## Perché è importante?

Finora hai lavorato con dati che avevi già salvati: un CSV, un dizionario scritto a mano, un file che ti hanno passato. Ma la maggior parte dei dati interessanti del mondo reale **non vive sul tuo disco** — vive sul server di un'altra azienda e cambia continuamente (il meteo, una quotazione, un risultato sportivo). Per ottenerli, devi richiederli al momento.

Qui entra in gioco un'**API** (Application Programming Interface): la porta che un servizio lascia aperta perché il tuo programma possa chiedergli dati senza dover capire come funziona internamente. Tu non sai come la NBA salva le sue statistiche né come una banca calcola il tasso di cambio — devi solo sapere **cosa chiedere** e **che formato di risposta aspettarti**.

Curiosità: hai già usato un'API senza saperlo. Pandas è una libreria che espone un'interfaccia (`.head()`, `.mean()`, `pd.DataFrame(...)`) senza che tu debba sapere come è risolta internamente. Consumare un'API su internet è esattamente la stessa idea, un gradino più in là.

## Come funziona HTTP: richiesta e risposta

Pensalo come ordinare al ristorante: tu (il **client**) dici al cameriere cosa vuoi in un formato che entrambi capite (la **request**), e il cameriere ti porta un piatto o ti avvisa che non c'è (la **response**). Questo è HTTP.

Tutta questa comunicazione viaggia su una **URL**, che si scompone in tre parti:

| Parte | Cos'è | Esempio |
|---|---|---|
| Schema | Il protocollo | `https://` |
| Indirizzo base | Dove vive il server | `official-joke-api.appspot.com` |
| Percorso | Quale risorsa chiedi nello specifico | `/jokes/ten` |

Quando quel percorso corrisponde a un'azione del server (non un file statico) si chiama **endpoint**.

La request porta un **metodo HTTP** che dice al server cosa vuoi fare:

- **GET**: richiedi dati (è quello che fa il tuo browser continuamente).
- **POST**: invii dati perché il server li elabori o li salvi.

La risposta porta un **codice di stato**:

| Intervallo | Significato | Esempio |
|---|---|---|
| 2xx | Successo | `200 OK` |
| 3xx | Redirect | La risorsa si è spostata |
| 4xx | Errore tuo | `404 Not Found` |
| 5xx | Errore del server | Qualcosa è fallito elaborando la tua richiesta |

Ed entrambi i lati portano **headers**: metadati sullo scambio (tipo di contenuto, data, codifica...), non il contenuto in sé.

In [ ]:
# Su Google Colab requests è già installato.
# Se sei in Jupyter locale puro e non ce l'hai, decommenta la riga seguente:
# !pip install requests

import requests
import pandas as pd

## GET con `requests`: la prima richiesta reale

Chiederemo dati a un'API pubblica che non richiede chiave di autenticazione: la **Official Joke API**, che restituisce barzellette in inglese in formato JSON.

`requests.get(url)` fa la richiesta e ti restituisce un oggetto risposta (qui lo chiamiamo `r`) con tutto lo scambio: codice di stato, headers, corpo.

Dato che a lezione dipendiamo dalla rete, avvolgiamo la richiesta in un `try/except`: se qualcosa fallisce (senza wifi, l'API è down), non vogliamo che il notebook si blocchi — proseguiamo con un avviso.

In [ ]:
url_chistes = "https://official-joke-api.appspot.com/jokes/ten"

try:
    r = requests.get(url_chistes, timeout=10)
    print("status_code:", r.status_code)
except requests.exceptions.RequestException as e:
    r = None
    print("Non è stato possibile connettersi all'API:", e)
    print("Proseguiamo con dati di esempio più sotto.")

## Headers: request e response

`r.request` conserva la richiesta che hai mandato tu (con i suoi headers e, in un GET, senza body). `r.headers` conserva ciò che ha risposto il server — si comporta come un dizionario Python.

In [ ]:
if r is not None:
    print("Headers che HAI MANDATO tu (request):")
    print(r.request.headers)
    print()
    print("Body della request (in un GET normale, non c'è body):", r.request.body)
    print()
    print("Headers che HA MANDATO il server (response):")
    print(r.headers)
    print()
    print("Content-Type della risposta:", r.headers.get("Content-Type"))
else:
    print("Senza connessione: non c'è nessun oggetto risposta da ispezionare in questa esecuzione.")

## Da JSON a DataFrame: `.json()` + `pd.DataFrame()`

Il metodo `.json()` dell'oggetto risposta converte il corpo (testo JSON) direttamente in liste/dizionari Python — equivale a fare `json.loads(r.text)`, ma in un solo passaggio.

Come buona pratica, **controlla sempre lo `status_code` prima di fidarti dei dati**: se la richiesta è fallita, `.json()` potrebbe non contenere quello che ti aspetti (o dare direttamente errore). Per questo, se `r` è `None` o lo `status_code` non è 200, usiamo un JSON di riserva (`fallback`) hardcodato — così il notebook funziona comunque anche senza internet al momento dell'esecuzione.

In [ ]:
# JSON di riserva in caso fallisca la connessione durante la demo
chistes_fallback = [
    {"type": "general", "setup": "Perché le foche del circo guardano sempre in alto?",
     "punchline": "Perché lì ci sono gli addestratori.", "id": 1},
    {"type": "general", "setup": "Cosa dice un semaforo a un altro?",
     "punchline": "Non guardarmi, mi sto cambiando.", "id": 2},
    {"type": "programming", "setup": "Quanti programmatori servono per cambiare una lampadina?",
     "punchline": "Nessuno, è un problema di hardware.", "id": 3},
]

if r is not None and r.status_code == 200:
    chistes = r.json()
    print("Dati ottenuti in diretta dall'API ✅")
else:
    chistes = chistes_fallback
    print("Utilizzo dati di esempio (fallback) ⚠️")

chistes[:2]

In [ ]:
df_chistes = pd.DataFrame(chistes)
df_chistes.head()

## JSON annidato: `pd.json_normalize()`

Il caso precedente era un JSON **piatto**: ogni barzelletta è un dizionario semplice, quindi `pd.DataFrame()` lo risolve direttamente. Ma molte API restituiscono JSON **annidato** — un dizionario dentro un altro. Qui `pd.DataFrame()` non basta: la colonna annidata rimane come colonna di dizionari, inutile per filtrare.

Esempio reale: l'API di **Fruityvice** restituisce, per ogni frutto, un dizionario `nutritions` con le proprie chiavi (`calories`, `fat`, `sugar`...).

In [ ]:
# JSON di riserva (struttura identica a quella reale, annidata) in caso fallisca la connessione
frutas_fallback = [
    {"name": "Cherry", "id": 1, "family": "Rosaceae", "genus": "Prunus", "order": "Rosales",
     "nutritions": {"calories": 50, "fat": 1.0, "sugar": 8.0, "carbohydrates": 12.0, "protein": 1.0}},
    {"name": "Banana", "id": 2, "family": "Musaceae", "genus": "Musa", "order": "Zingiberales",
     "nutritions": {"calories": 96, "fat": 0.2, "sugar": 17.2, "carbohydrates": 22.0, "protein": 1.0}},
]

try:
    r_frutas = requests.get("https://fruityvice.com/api/fruit/all", timeout=10)
    if r_frutas.status_code == 200:
        frutas = r_frutas.json()
        print("Dati in diretta ✅")
    else:
        frutas = frutas_fallback
        print(f"L'API ha risposto {r_frutas.status_code}, uso il fallback ⚠️")
except requests.exceptions.RequestException as e:
    frutas = frutas_fallback
    print("Non è stato possibile connettersi, uso il fallback ⚠️:", e)

In [ ]:
# Se lo passiamo direttamente a DataFrame, 'nutritions' rimane come colonna di dizionari
df_frutas_plano = pd.DataFrame(frutas)
df_frutas_plano.head()

In [ ]:
# pd.json_normalize() appiattisce i dizionari annidati in colonne proprie:
# 'nutritions' -> 'nutritions.calories', 'nutritions.fat', ecc.
df_frutas = pd.json_normalize(frutas)
df_frutas.head()

In [ ]:
# Con il JSON già appiattito, filtrare è diretto come con qualsiasi DataFrame
cereza = df_frutas.loc[df_frutas["name"] == "Cherry"]
cereza.iloc[0]["family"], cereza.iloc[0]["nutritions.calories"]

**Regola pratica:** se il JSON è piatto, `pd.DataFrame()` basta. Se ha dizionari annidati dentro le chiavi, usa `pd.json_normalize()` fin dall'inizio — ti risparmia di smontare la struttura a mano con dei cicli.

## GET con parametri e POST: inviare informazioni

Un GET può portare parametri nella URL (la *query string*: tutto dopo il `?`). In Python non costruisci quella stringa a mano: passi un dizionario all'argomento `params` di `requests.get()`.

Il metodo **POST**, invece, manda i dati nel **corpo** della richiesta (`data=`), non nella URL — si usa tipicamente per creare o inviare qualcosa (un modulo, un nuovo record).

Usiamo `httpbin.org`, un servizio pensato apposta per testare le richieste: ti restituisce, in JSON, esattamente quello che gli hai mandato.

In [ ]:
url_get = "http://httpbin.org/get"
url_post = "http://httpbin.org/post"
payload = {"name": "Ada", "curso": "Python"}

try:
    r_get = requests.get(url_get, params=payload, timeout=10)
    r_post = requests.post(url_post, data=payload, timeout=10)

    print("URL del GET (i parametri vanno nella URL):")
    print(r_get.url)
    print()
    print("Body del GET (non dovrebbe esserci body):", r_get.request.body)
    print()
    print("Body del POST (i dati vanno nel corpo):", r_post.request.body)
    print()
    print("Quello che il server ha ricevuto come 'args' del GET:", r_get.json()["args"])
    print("Quello che il server ha ricevuto come 'form' del POST:", r_post.json()["form"])
except requests.exceptions.RequestException as e:
    print("Non è stato possibile connettersi a httpbin.org in questo momento:", e)
    print("(L'idea da portarti a casa: GET manda i dati nella URL tramite 'params', POST li manda nel corpo tramite 'data'.)")

## Riepilogo: cosa porti a casa da questa lezione

- Ogni API segue il ciclo HTTP di **request** (richiesta) e **response** (risposta): metodo + URL da un lato, codice di stato + headers + body dall'altro.
- **GET** per leggere dati (con `params` opzionali); **POST** per inviare dati che il server elaborerà (con `data`).
- Controlla sempre `r.status_code` prima di fidarti di `r.json()` — 2xx è successo, 4xx è errore tuo, 5xx è errore del server.
- `.json()` converte il body in strutture Python. `pd.DataFrame()` basta se il JSON è piatto; `pd.json_normalize()` lo appiattisce quando arriva annidato.
- Avvolgere le richieste in `try/except requests.exceptions.RequestException` evita che un problema di rete blocchi tutto il programma.

### Per continuare a esplorare

Cerca "free public APIs no auth needed" e prova un'altra API che non richieda chiave — lo schema (`requests.get()` → controllo `status_code` → `.json()` → `pd.DataFrame()` o `pd.json_normalize()`) si ripete sempre.